## Importing Libraries

In [9]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

load_dotenv()

llm = ChatOpenAI(model="gpt-5-nano", temperature=0.0)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

### STEP 0: CREATE A SAMPLE DOCUMENT
*In real RAG you'd load PDFs, web pages, etc.*

*We create a text file about backend concepts to query against.*

In [10]:
sample_content = """
PostgreSQL Indexing Guide

B-tree Indexes
B-tree is the default index type in PostgreSQL. It works for equality and range queries.
B-tree indexes support operators: =, <, >, <=, >=, BETWEEN, IN, IS NULL, LIKE 'prefix%'.
Create with: CREATE INDEX idx_name ON table(column);

Composite Indexes
A composite index covers multiple columns. Column order matters significantly.
The leftmost column must appear in the query WHERE clause for the index to be used.
Example: CREATE INDEX idx_user_tenant ON notes(user_id, tenant_id);
This index helps queries filtering by user_id alone OR user_id + tenant_id together.
It does NOT help queries filtering only by tenant_id.

Partial Indexes
A partial index only indexes rows matching a condition.
Smaller and faster than full indexes when you query a subset of rows.
Example: CREATE INDEX idx_active_users ON users(email) WHERE is_active = true;
Only active users are indexed — perfect if 90% of queries filter on active users.

EXPLAIN ANALYZE
Use EXPLAIN ANALYZE to understand query execution plans.
Seq Scan = full table scan (bad for large tables).
Index Scan = uses index (good).
Bitmap Index Scan = combines multiple indexes (good for complex conditions).
Always run EXPLAIN ANALYZE before and after adding indexes to measure impact.

Redis Caching Patterns
Cache-Aside (Lazy Loading): Application checks cache first. On miss, reads DB, writes to cache.
Write-Through: Write to cache and DB simultaneously. Cache always consistent with DB.
Write-Behind: Write to cache immediately, async write to DB. Fast but risk of data loss.
TTL (Time To Live): Always set expiry on cached data to prevent stale data issues.

JWT Authentication
JWT = JSON Web Token. Stateless authentication — server doesn't store sessions.
Structure: header.payload.signature (base64 encoded, dot separated).
Access tokens: short-lived (15–60 min). Refresh tokens: long-lived (7–30 days).
Never store sensitive data in JWT payload — it's base64 encoded, not encrypted.
Always validate: signature, expiry (exp claim), token type to prevent swapping attacks.
"""

with open("backend_knowledge.txt", "w") as f:
    f.write(sample_content)

### STEP 1: LOAD DOCUMENTS

In [11]:
print("=== Loading Document ===")
loader = TextLoader("backend_knowledge.txt")
docs = loader.load()
print(f"Loaded {len(docs)} document(s).")
print(f"Total characters: {len(docs[0].page_content)}")
print(f"Metadata: {docs[0].metadata}")
print()

=== Loading Document ===
Loaded 1 document(s).
Total characters: 2074
Metadata: {'source': 'backend_knowledge.txt'}



### STEP 2: SPLIT INTO CHUNKS 

In [12]:
print("=== Step 2: Splitting Document ===")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, 
    chunk_overlap=30,
    separators=["\n\n", "\n", ".", " ", "" ]
    )
chunks = splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks.")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Characters: {len(chunk.page_content)}")
    print(f"Content:\n{chunk.page_content}")
print()

=== Step 2: Splitting Document ===
Created 11 chunks.

--- Chunk 1 ---
Characters: 272
Content:
PostgreSQL Indexing Guide

B-tree Indexes
B-tree is the default index type in PostgreSQL. It works for equality and range queries.
B-tree indexes support operators: =, <, >, <=, >=, BETWEEN, IN, IS NULL, LIKE 'prefix%'.
Create with: CREATE INDEX idx_name ON table(column);

--- Chunk 2 ---
Characters: 248
Content:
Composite Indexes
A composite index covers multiple columns. Column order matters significantly.
The leftmost column must appear in the query WHERE clause for the index to be used.
Example: CREATE INDEX idx_user_tenant ON notes(user_id, tenant_id);

--- Chunk 3 ---
Characters: 138
Content:
This index helps queries filtering by user_id alone OR user_id + tenant_id together.
It does NOT help queries filtering only by tenant_id.



### STEP 3: EMBED + STORE IN VECTOR DB
*Chroma is in-memory by default — persists to disk if you pass persist_directory*

*persist_directory="./chroma_db"  # uncomment to persist to disk*

In [13]:
!pip install chromadb


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
print("=== Step 3: Embed and Store ===")
# Chroma is in-memory by default — persists to disk if you pass persist_directory
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    # persist_directory="./chroma_db"  # uncomment to persist to disk
)
print(f"Vector store created with {vectorstore._collection.count()} vectors")
print()

=== Step 3: Embed and Store ===
Vector store created with 11 vectors



### STEP 4: RETRIVE

In [15]:
print("=== Step 4: Retrieve ===")
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

test_query = "What is a composite index and how does column order affect it?"
retrieved_docs = retriever.invoke(test_query)
print(f"Retrieved {len(retrieved_docs)} chunks for query: '{test_query}'")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n  Chunk {i}: {doc.page_content[:120]}...")
print()

=== Step 4: Retrieve ===
Retrieved 3 chunks for query: 'What is a composite index and how does column order affect it?'

  Chunk 1: Composite Indexes
A composite index covers multiple columns. Column order matters significantly.
The leftmost column mus...

  Chunk 2: EXPLAIN ANALYZE
Use EXPLAIN ANALYZE to understand query execution plans.
Seq Scan = full table scan (bad for large table...

  Chunk 3: Partial Indexes
A partial index only indexes rows matching a condition.
Smaller and faster than full indexes when you qu...



### STEP 5: FULL RAG CHAIN

In [16]:
print("=== Step 5: Full RAG Chain ===")

def format_docs_for_prompt(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a backend engineering expert assistant.
Answer questions using ONLY the context provided below.
If the context doesn't contain the answer, say exactly: "I don't have that information."
Do not use any outside knowledge.

Context:
{context}"""),
    ("human", "{question}"),
])

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs_for_prompt),
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Test with several questions
questions = [
    "What is a composite index and how does column order affect it?",
    "What are the different Redis caching patterns?",
    "How should I use EXPLAIN ANALYZE?",
    "What is the difference between access tokens and refresh tokens?",
    "What is the capital of France?",  # should say "I don't have that information"
]

for q in questions:
    print(f"\nQuestion: {q}")
    answer = rag_chain.invoke(q)
    print(f"Answer: {answer}")
    print("-" * 50)

os.remove("backend_knowledge.txt")


=== Step 5: Full RAG Chain ===

Question: What is a composite index and how does column order affect it?
Answer: - A composite index is an index that covers multiple columns.

- Column order matters significantly. The order determines how the index can be used in a query.

- The leftmost column must appear in the query WHERE clause for the composite index to be used.

- Example: CREATE INDEX idx_user_tenant ON notes(user_id, tenant_id); 
  - This index is usable when the query constrains user_id (and can also help if it constrains tenant_id as well, depending on the query). 
  - If the query does not include user_id in the WHERE clause, this index cannot be used.
--------------------------------------------------

Question: What are the different Redis caching patterns?
Answer: - Cache-Aside (Lazy Loading): Application checks cache first. On miss, reads DB, writes to cache.
- Write-Through: Write to cache and DB simultaneously. Cache always consistent with DB.
- Write-Behind: Write to 